## Scenario 2: A cross-functional team with one data scientist working on an ML model

This scenario demonstrates how a data scientist can use MLflow to track machine learning experiments in a team setting, using a centralized tracking server. This setup is common in organizations where multiple people need to access experiment results, models, and artifacts.

### MLflow setup overview:
- **Tracking server:** Yes (runs as a local server, accessible to the team)
- **Backend store:** SQLite database (stores experiment metadata in `backend.db`)
- **Artifacts store:** Local filesystem (stores model files and other artifacts)

With this setup, all experiment runs, parameters, metrics, and artifacts are saved in a central location. Team members can explore and compare experiments using the MLflow UI, even from different machines (if the server is accessible).

### How to use the MLflow tracking server and UI
- **First, you must launch the MLflow tracking server** by running the following command in your terminal:
  ```bash
  mlflow server --backend-store-uri sqlite:///backend.db
  ```
- The UI will be available at the address printed in your terminal (by default, [http://localhost:5000](http://localhost:5000)).
- If you run the server on a remote machine or a different port, use the appropriate address (e.g., `http://<your-server>:<port>`).
- Use the UI to browse experiments, compare runs, and inspect logged models and artifacts.
- You can also interact with the model registry for collaborative model management.

> **Tip:** This setup is ideal for small teams and collaborative projects. For larger teams or production, you may use a remote database and cloud storage for the backend and artifacts.

In [9]:
import mlflow


mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [10]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://127.0.0.1:5000'


In [11]:
mlflow.search_experiments()

[<Experiment: artifact_location='file:///Users/oricardomg/Documents/Documentos%20Master/2nd%20Term/MLOPS/ie-mlops-nyc-taxis/03-experiment-tracking/mlruns/760520559899351722', creation_time=1761734649526, experiment_id='760520559899351722', last_update_time=1761734649526, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='file:///Users/oricardomg/Documents/Documentos%20Master/2nd%20Term/MLOPS/ie-mlops-nyc-taxis/03-experiment-tracking/mlruns/0', creation_time=1761734527776, experiment_id='0', last_update_time=1761734527776, lifecycle_stage='active', name='Default', tags={}>]

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, confusion_matrix
import numpy as np
import mlflow
import sklearn
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

mlflow.set_experiment("my-experiment-1")

X, y = load_iris(return_X_y=True)
class_names = load_iris().target_names

# Try several values of C to demonstrate experiment tracking
for C in [0.01, 0.1, 1, 10]:
    params = {"C": C, "random_state": 42, "max_iter": 1000, "solver": "lbfgs"}
    with mlflow.start_run() as run:
        mlflow.log_params(params)
        mlflow.log_param("sklearn_version", sklearn.__version__)
        lr = LogisticRegression(**params).fit(X, y)
        y_pred = lr.predict(X)
        acc = accuracy_score(y, y_pred)
        mlflow.log_metric("accuracy", acc)
        # Log model coefficients as params (flattened for logging)
        for i, coef in enumerate(lr.coef_.flatten()):
            mlflow.log_param(f"coef_{i}", coef)
        # Labeled confusion matrix as DataFrame
        cm = confusion_matrix(y, y_pred)
        cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
        cm_df.to_csv("confusion_matrix_labeled.csv")
        mlflow.log_artifact("confusion_matrix_labeled.csv")
        # Confusion matrix heatmap as image
        plt.figure(figsize=(5,4))
        sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
        plt.title(f"Confusion Matrix (C={C})")
        plt.ylabel("True label")
        plt.xlabel("Predicted label")
        plt.tight_layout()
        plt.savefig("confusion_matrix_heatmap.png")
        plt.close()
        mlflow.log_artifact("confusion_matrix_heatmap.png")
        # Provide input_example and use 'name' instead of deprecated 'artifact_path'
        input_example = np.expand_dims(X[0], axis=0)
        mlflow.sklearn.log_model(lr, name="models", input_example=input_example)
        # Log model type, number of classes, and timestamp as tags
        mlflow.set_tag("model_type", type(lr).__name__)
        mlflow.set_tag("n_classes", len(np.unique(y)))
        mlflow.set_tag("run_time", datetime.datetime.now().isoformat())
        mlflow.set_tag("description", "Logistic regression on Iris dataset with varying C")
        print(f"Logged run for C={C}, accuracy={acc:.3f}")

Logged run for C=0.01, accuracy=0.873
🏃 View run silent-skink-248 at: http://127.0.0.1:5000/#/experiments/760520559899351722/runs/c586c66be54e416ab4e7ef311ad2a0d2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/760520559899351722
Logged run for C=0.1, accuracy=0.960
🏃 View run mercurial-wolf-677 at: http://127.0.0.1:5000/#/experiments/760520559899351722/runs/4dcddeaf07d94de3b3228cc44a9f1edb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/760520559899351722
Logged run for C=1, accuracy=0.973
🏃 View run popular-fly-792 at: http://127.0.0.1:5000/#/experiments/760520559899351722/runs/b5b19526f8384ea6baaa897e82220803
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/760520559899351722
Logged run for C=10, accuracy=0.980
🏃 View run illustrious-wasp-983 at: http://127.0.0.1:5000/#/experiments/760520559899351722/runs/0115627eaa8247fa8850ad4377da0e11
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/760520559899351722


In [13]:
mlflow.search_experiments()

[<Experiment: artifact_location='file:///Users/oricardomg/Documents/Documentos%20Master/2nd%20Term/MLOPS/ie-mlops-nyc-taxis/03-experiment-tracking/mlruns/760520559899351722', creation_time=1761734649526, experiment_id='760520559899351722', last_update_time=1761734649526, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='file:///Users/oricardomg/Documents/Documentos%20Master/2nd%20Term/MLOPS/ie-mlops-nyc-taxis/03-experiment-tracking/mlruns/0', creation_time=1761734527776, experiment_id='0', last_update_time=1761734527776, lifecycle_stage='active', name='Default', tags={}>]

### Interacting with the model registry

In [14]:
from mlflow.tracking import MlflowClient


client = MlflowClient("http://127.0.0.1:5000")

In [15]:
client.search_registered_models()

[]

In [16]:
run_id = client.search_runs(experiment_ids='1')[0].info.run_id
mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name='iris-classifier'
)

IndexError: list index out of range